# 1. Импорт библиотек

In [1]:
import os
import sys
sys.path.append('../')

import pandas as pd
pd.set_option('display.max_columns', 450)

import numpy as np

from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score

from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

import warnings
warnings.filterwarnings("ignore")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
print(f"device: {torch.cuda.get_device_name(0)}")

device: Tesla T4


In [3]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
set_seed(42)

# 2. Загрузка данных

In [4]:
train_df = pd.read_parquet('/content/drive/MyDrive/data_project/train_collab.pqt')
test_df = pd.read_parquet('/content/drive/MyDrive/data_project/test_collab.pqt')
oot_df = pd.read_parquet('/content/drive/MyDrive/data_project/oot_collab.pqt')

print(train_df.shape)
print(test_df.shape)
print(oot_df.shape)

(350849, 458)
(150365, 458)
(89326, 458)


# 3. Подготовка данных

In [5]:
STAT_COLS = ['TransactionID', 'target', 'TransactionDT', 'card1']

# 3.1 Создание последовательностей

In [6]:
feature_cols = [c for c in train_df.columns if c not in STAT_COLS]

cat_features = [col for col in feature_cols if '_idx' in col]
num_features = [col for col in feature_cols if col not in cat_features]

# отдельно категориальные для эмбедингов
vocab_sizes = {}

for col in cat_features:
    # 
    max_idx = int(train_df[col].max())
    vocab_sizes[col] = max_idx + 1


# 
train_df = train_df.sort_values(['card1', 'TransactionDT'])
test_df = test_df.sort_values(['card1', 'TransactionDT'])
oot_df = oot_df.sort_values(['card1', 'TransactionDT'])

# последовательности для train
train_sequences = []
train_lengths = []
train_targets = []
train_ids = []

for card, group in train_df.groupby('card1'):
    seq = group[feature_cols].values.astype(np.float32)
    train_sequences.append(seq)
    train_lengths.append(len(seq))
    train_targets.append(group['target'].values.astype(np.float32))
    train_ids.append(group['TransactionID'].values) 

# последовательности для test
test_sequences = []
test_lengths = []
test_targets = []
test_ids = []

for card, group in test_df.groupby('card1'):
    seq = group[feature_cols].values.astype(np.float32)
    test_sequences.append(seq)
    test_lengths.append(len(seq))
    test_targets.append(group['target'].values.astype(np.float32))
    test_ids.append(group['TransactionID'].values)

# последовательности для oot
oot_sequences = []
oot_lengths = []
oot_targets = []
oot_ids = []

for card, group in oot_df.groupby('card1'):
    seq = group[feature_cols].values.astype(np.float32)
    oot_sequences.append(seq)
    oot_lengths.append(len(seq))
    oot_targets.append(group['target'].values.astype(np.float32))
    oot_ids.append(group['TransactionID'].values)

In [7]:
# check

print(f"train: {len(train_sequences)} карт, макс длина: {max(train_lengths)}, медиана: {round(np.median(train_lengths), 0)}")
print(f"test: {len(test_sequences)} карт, макс длина: {max(test_lengths)}, медиана: {round(np.median(test_lengths), 0)}")
print(f"oot: {len(oot_sequences)} карт, макс длина: {max(oot_lengths)}, медиана: {round(np.median(oot_lengths), 0)}")
print(f"features: {len(feature_cols)}")

train: 11740 карт, макс длина: 8786, медиана: 3.0
test: 8924 карт, макс длина: 3707, медиана: 2.0
oot: 6334 карт, макс длина: 2439, медиана: 2.0
features: 454


# 3.2 Датасет + паддинг

In [8]:
# Dataset
class PDataset(Dataset):
    def __init__(self, sequences, targets, ids):
        self.sequences = [torch.FloatTensor(seq) for seq in sequences]
        self.targets = [torch.FloatTensor(tgt) for tgt in targets]
        self.ids = [torch.LongTensor(ids_seq) for ids_seq in ids]
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx], self.ids[idx]

# 
def collate_fn(batch):
    sequences = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    ids = [item[2] for item in batch]
    lengths = [len(seq) for seq in sequences]
    
    # Паддинг до макс длины в батче
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    targets_padded = pad_sequence(targets, batch_first=True, padding_value=0)
    ids_padded = pad_sequence(ids, batch_first=True, padding_value=0)
    lengths_tensor = torch.tensor(lengths)
    
    return sequences_padded, targets_padded, ids_padded, lengths_tensor

# 
train_dataset = PDataset(train_sequences, train_targets, train_ids)
test_dataset = PDataset(test_sequences, test_targets, test_ids)
oot_dataset = PDataset(oot_sequences, oot_targets, oot_ids)

# 
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
oot_loader = DataLoader(oot_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

# 3.3 Model

In [9]:
class GRUModel(nn.Module):
    def __init__(self, num_numeric, cat_features, vocab_sizes, emb_dim=16, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        
        self.cat_features = cat_features
        self.num_numeric = num_numeric
        
        # 
        self.embeddings = nn.ModuleList()
        total_emb_dim = 0
        
        for col in cat_features:
            vocab_size = vocab_sizes[col]

            emb_d = emb_dim
            self.embeddings.append(nn.Embedding(vocab_size, emb_d, padding_idx=0))
            total_emb_dim += emb_d
        
        # Общая размерность входа для GRU (числовые + все эмбеддинги)
        gru_input_dim = num_numeric + total_emb_dim
        
        # GRU
        self.gru = nn.GRU(
            input_size=gru_input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=False
        )
        
        # Выходной слой
        self.fc = nn.Linear(hidden_dim, 1)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, lengths):
        """
        x: (batch, seq_len, total_features)
        lengths: (batch,) - реальные длины последовательностей (без паддинга)
        """
        
        batch_size, seq_len, _ = x.shape
        
        numeric_x = x[:, :, :self.num_numeric]  # (batch, seq_len, num_numeric)
        
        # 
        cat_x = x[:, :, self.num_numeric:]  # (batch, seq_len, num_cat)
        cat_x = cat_x.long()  # для Embedding нужен long
        
        # 
        cat_embeds = []
        for i, emb_layer in enumerate(self.embeddings):

            col_data = cat_x[:, :, i]  # (batch, seq_len)
            embedded = emb_layer(col_data)  # (batch, seq_len, emb_dim)
            cat_embeds.append(embedded)
        
        # 
        cat_concat = torch.cat(cat_embeds, dim=-1)  # (batch, seq_len, total_emb_dim)
        
        # числовые и категориальные
        combined = torch.cat([numeric_x, cat_concat], dim=-1)  # (batch, seq_len, gru_input_dim)
       
        lengths = lengths.cpu()
        
        #
        packed = nn.utils.rnn.pack_padded_sequence(combined, lengths, batch_first=True, enforce_sorted=False)
        
        # GRU
        packed_output, hidden = self.gru(packed)
        
        # unpack обратно
        output, _ = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True, total_length=seq_len)
        
        # Выходной слой для каждого шага
        output = self.dropout(output)
        
        logits = self.fc(output)  # (batch, seq_len, 1)
        
        return logits

In [10]:
# validation
def evaluate_model(model, data_loader, device, criterion=None):
    model.eval()
    all_preds = []
    all_targets = []
    total_loss = 0
    
    with torch.no_grad():
        for sequences, targets, ids, lengths in data_loader:
            sequences = sequences.to(device)
            targets = targets.to(device)
            lengths = lengths.to(device)
            
            logits = model(sequences, lengths)
            logits = logits.squeeze(-1)  # (batch, seq_len)
            
            if criterion is not None:
                loss = 0
                for i, length in enumerate(lengths):
                    loss += criterion(logits[i, :length], targets[i, :length])
                total_loss += loss.item() / len(lengths)
            
            for i, length in enumerate(lengths):
                preds = torch.sigmoid(logits[i, :length]).cpu().numpy()
                targs = targets[i, :length].cpu().numpy()
                all_preds.extend(preds)
                all_targets.extend(targs)
    
    auc_roc = roc_auc_score(all_targets, all_preds)
    
    return {
        'auc_roc': auc_roc,
        'loss': total_loss / len(data_loader) if criterion else None
    }

In [11]:
# model
num_numeric = len(num_features) 
cat_features = cat_features
vocab_sizes = vocab_sizes

#
model = GRUModel(
    num_numeric=num_numeric,
    cat_features=cat_features,
    vocab_sizes=vocab_sizes,
    emb_dim=16,
    hidden_dim=128,
    num_layers=2,
    dropout=0.4
)

#
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# 
total_params = sum(p.numel() for p in model.parameters())
print(f"Параметров: {total_params}")

# 
all_targets = []
for targets_seq in train_targets:
    all_targets.extend(targets_seq)


all_targets = np.array(all_targets)
n_neg = (all_targets == 0).sum()
n_pos = (all_targets == 1).sum()

pos_weight = n_neg / n_pos

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))

optimizer = optim.Adam(model.parameters(), lr=0.001)

Параметров: 417249


In [ ]:
# Обучение
num_epochs = 15

with tqdm(total=num_epochs, desc='Training') as pbar:
    
    for epoch in range(num_epochs):
        # train
        model.train()
        train_loss = 0
        train_preds = []
        train_targets_all = []
        
        #
        for sequences, targets, ids, lengths in train_loader:
            sequences = sequences.to(device)
            targets = targets.to(device)
            #lengths = lengths.to(device)
            
            logits = model(sequences, lengths)
            logits = logits.squeeze(-1)
            
            loss = 0
            for i, length in enumerate(lengths):
                loss += criterion(logits[i, :length], targets[i, :length])
            loss = loss / len(lengths)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
            for i, length in enumerate(lengths):
                preds = torch.sigmoid(logits[i, :length]).detach().cpu().numpy()
                targs = targets[i, :length].cpu().numpy()
                train_preds.extend(preds)
                train_targets_all.extend(targs)
        
        # 
        train_auc_roc = roc_auc_score(train_targets_all, train_preds)
        avg_train_loss = train_loss / len(train_loader)
        
        test_metrics = evaluate_model(model, test_loader, device, criterion)
        oot_metrics = evaluate_model(model, oot_loader, device, criterion)
        
        pbar.update(1)
        

        # Вывод результатов
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"  train - loss: {avg_train_loss:.4f}, AUC: {train_auc_roc:.4f}")
        print(f"  test  - loss: {test_metrics['loss']:.4f}, AUC: {test_metrics['auc_roc']:.4f}")
        print(f"  oot   - loss: {oot_metrics['loss']:.4f}, AUC: {oot_metrics['auc_roc']:.4f}")
        print("-" * 35)


Training:   0%|          | 0/15 [00:00<?, ?it/s]


Epoch 1/15
  train - loss: 1.0649, AUC: 0.7418
  test  - loss: 1.0622, AUC: 0.8013
  oot   - loss: 0.9646, AUC: 0.7860
-----------------------------------

Epoch 2/15
  train - loss: 0.9780, AUC: 0.7789
  test  - loss: 0.9377, AUC: 0.8049
  oot   - loss: 0.8452, AUC: 0.7917
-----------------------------------

Epoch 3/15
  train - loss: 0.9353, AUC: 0.7871
  test  - loss: 0.9170, AUC: 0.8064
  oot   - loss: 0.8280, AUC: 0.7927
-----------------------------------


In [ ]:
ssss

# 4. Сохранение данных

In [ ]:
def get_predictions_with_label(model, data_loader, device, label):
    model.eval()
    results = []
    
    with torch.no_grad():
        for sequences, targets, ids, lengths in data_loader:
            sequences = sequences.to(device)
            lengths = lengths.to(device)
            
            logits = model(sequences, lengths)
            logits = logits.squeeze(-1)
            probs = torch.sigmoid(logits)
            
            for i, length in enumerate(lengths):
                preds = probs[i, :length].cpu().numpy()
                trans_ids = ids[i, :length].cpu().numpy()
                targs = targets[i, :length].cpu().numpy()
                
                for tid, prob, targ in zip(trans_ids, preds, targs):
                    results.append({
                        'TransactionID': tid,
                        'pred_prob': prob,
                        'target': targ,
                        'train_test_oot': label
                    })
    
    return pd.DataFrame(results)

# Получаем предсказания для всех выборок
train_preds = get_predictions_with_label(model, train_loader, device, 'train')
test_preds = get_predictions_with_label(model, test_loader, device, 'test')
oot_preds = get_predictions_with_label(model, oot_loader, device, 'oot')

# Объединяем
final_df = pd.concat([train_preds, test_preds, oot_preds], ignore_index=True)

In [ ]:
final_df.shape

(185187, 4)

In [ ]:
train_df.shape

,TransactionID,target,TransactionDT,card1,C1,C10,C11,C12,C13,C14,C2,C4,C5,C6,C7,C8,C9,D1,D10,D11,D12,D13,D14,D15,D2,D3,D4,D5,D6,D7,D8,D9,TransactionAmt,V10,V100,V101,V102,V103,V104,V105,V106,V109,V11,V115,V116,V12,V123,V124,V125,V126,V127,V128,V129,V13,V130,V131,V132,V133,V134,V135,V136,V137,V138,V139,V140,V141,V142,V143,V144,V145,V146,V147,V148,V149,V15,V150,V151,V152,V153,V154,V155,V156,V157,V158,V159,V16,V160,V161,V162,V163,V164,V165,V166,V167,V168,V169,V17,V170,V171,V172,V173,V174,V175,V176,V177,V178,V179,V18,V180,V181,V182,V183,V184,V185,V186,V187,V188,V189,V19,V190,V191,V192,V193,V194,V195,V196,V197,V198,V199,V2,V20,V200,V201,V202,V203,V204,V205,V206,V207,V208,V209,V21,V210,V211,V212,V213,V214,V215,V216,V217,V218,V219,V22,V220,V221,V222,V223,V224,V225,V226,V227,V228,V229,V23,V230,V231,V232,V233,V234,V235,V236,V237,V238,V239,V24,V242,V243,V244,V245,V246,V247,V248,V249,V25,V250,V251,V252,V253,V254,V255,V256,V257,V258,V259,V26,V260,V261,V262,V263,V264,V265,V266,V267,V268,V269,V270,V271,V272,V273,V274,V275,V276,V277,V278,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,...,V297,V298,V299,V3,V30,V300,V301,V302,V303,V304,V306,V307,V308,V309,V31,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V32,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V33,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339,V34,V35,V36,V37,V38,V39,V4,V40,V42,V43,V44,V45,V46,V47,V48,V49,V5,V50,V51,V52,V53,V54,V55,V56,V57,V58,V59,V6,V60,V61,V62,V63,V64,V66,V67,V69,V7,V70,V71,V72,V73,V74,V75,V76,V77,V78,V79,V8,V80,V81,V82,V83,V84,V85,V86,V87,V9,V90,V91,V92,V93,V94,V95,V96,V97,V98,V99,addr1,addr1_new,addr2_new,amt_max_168h,amt_max_1h,amt_max_24h,amt_max_6h,amt_max_72h,amt_mean_168h,amt_mean_1h,amt_mean_24h,amt_mean_6h,amt_mean_72h,amt_median_168h,amt_median_1h,amt_median_24h,amt_median_6h,amt_median_72h,amt_min_168h,amt_min_1h,amt_min_24h,amt_min_6h,amt_min_72h,amt_ratio_to_mean_168h,amt_ratio_to_mean_1h,amt_ratio_to_mean_24h,amt_ratio_to_mean_6h,amt_ratio_to_mean_72h,amt_ratio_to_median_168h,amt_ratio_to_median_1h,amt_ratio_to_median_24h,amt_ratio_to_median_6h,amt_ratio_to_median_72h,amt_std_168h,amt_std_1h,amt_std_24h,amt_std_6h,amt_std_72h,amt_sum_168h,amt_sum_1h,amt_sum_24h,amt_sum_6h,amt_sum_72h,card_addr1_changed,card_addr2_changed,card_amt_is_max,card_amt_max,card_amt_mean,card_amt_median,card_amt_min,card_amt_more_2x_mean,card_amt_more_3x_mean,card_amt_ratio_to_mean,card_amt_ratio_to_median,card_amt_std,card_hour_changed,card_mode_addr1,card_mode_addr2,card_mode_hour,card_night_ratio,card_unique_P_email,card_unique_P_email_gr,card_unique_R_email,card_unique_R_email_gr,card_unique_addr1,card_unique_addr2,card_weekend_ratio,count_168h,count_1h,count_24h,count_6h,count_72h,dist1,dist2,hour,hour_cos,hour_sin,is_night,max_gap_168h,max_gap_1h,max_gap_24h,max_gap_6h,max_gap_72h,mean_gap_168h,mean_gap_1h,mean_gap_24h,mean_gap_6h,mean_gap_72h,min_gap_168h,min_gap_1h,min_gap_24h,min_gap_6h,min_gap_72h,minute,second,DeviceInfo_brand_idx,DeviceType_new_idx,M1_new_idx,M2_new_idx,M3_new_idx,M4_new_idx,M5_new_idx,M6_new_idx,M7_new_idx,M8_new_idx,M9_new_idx,P_emaildomain_grouped_idx,ProductCD_new_idx,R_emaildomain_grouped_idx,card4_new_idx,card6_new_idx
243924,3230924,0.0,5787419,1000,-0.090231,-0.038520,-0.097433,-0.039976,-0.236358,-0.144641,-0.085946,-0.048358,-0.215168,-0.111942,-0.034973,-0.047044,-0.265863,-0.545719,0.079470,-0.962955,2.666930,2.905462,2.741469,0.069144,-1.005709,-1.085754,0.373693,0.993224,2.478441,3.682374,2.179571,2.609020,-0.473530,-0.986335,0.010187,0.016926,0.005725,0.013385,0.018853,0.015664,0.017892,0.020902,-0.986335,0.020132,0.020863,0.396596,0.020173,0.017654,0.019471,-0.077592,-0.195650,-0.129173,-0.064665,0.396482,-0.283358,-0.173601,-0.075575,-0.107139,-0.099174,-0.043517,-0.067164,-0.05186,-0.409866,-0.409865,-0.409865,-0.409866,-0.409866,-0.409849,-0.409837,-0.408804,-0.409866,-0.409866,-0.409866,-0.409866,0.400736,-0.332233,-0.409804,-0.409747,-0.409866,-0.409866,-0.409866,-0.409866,-0.409866,-0.409866,-0.160818,0.400

In [ ]:
stop

NameError: name 'stop' is not defined

In [ ]:

final_df.to_parquet('content/drive/MyDrive/prodata_project/gru_preds.pqt')